In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType
)
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

SOURCE_PATH  = "abfss://raw@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"
TARGET_PATH = "abfss://bronzelayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"

PARQUET_COMPRESSION = "snappy"

RAW_SCHEMA = StructType([
    StructField("SNo",       LongType(),   nullable=True),
    StructField("Name",      StringType(), nullable=True),
    StructField("Symbol",    StringType(), nullable=True),
    StructField("Date",      StringType(), nullable=True),
    StructField("High",      DoubleType(), nullable=True),
    StructField("Low",       DoubleType(), nullable=True),
    StructField("Open",      DoubleType(), nullable=True),
    StructField("Close",     DoubleType(), nullable=True),
    StructField("Volume",    DoubleType(), nullable=True),
    StructField("Marketcap", DoubleType(), nullable=True),
])


def get_spark_session() -> SparkSession:
    return SparkSession.builder.getOrCreate()


def read_all_csvs(spark: SparkSession):
    log.info(f"Reading CSV files from: {SOURCE_PATH}")
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .schema(RAW_SCHEMA)
        .csv(SOURCE_PATH + "*.csv")
    )
    log.info(f"Raw rows loaded: {df.count():,}")
    return df


def add_metadata(df):

    return (
        df
        .withColumn("trade_date",
            F.to_timestamp(F.col("Date"), "yyyy-MM-dd HH:mm:ss"))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("data_source", F.lit("coinmarketcap_historical_batch"))
    )


def write_to_bronze(df) -> None:
    log.info(f"Writing Parquet to: {TARGET_PATH}")
    (
        df.write
        .mode("overwrite")
        .option("compression", PARQUET_COMPRESSION)
        .format("parquet")
        .partitionBy("Symbol")
        .save(TARGET_PATH)
    )
    log.info("Bronze Parquet written successfully!")


def main():
    log.info("BATCH INGESTION — Bronze Layer Started")
    spark = get_spark_session()

    df_raw      = read_all_csvs(spark)
    df_bronze   = add_metadata(df_raw)
    write_to_bronze(df_bronze)

    log.info("Verifying...")
    df_verify = spark.read.parquet(TARGET_PATH)
    log.info(f"Rows in Parquet: {df_verify.count():,}")
    log.info(f"Unique coins  : {df_verify.select('Name').distinct().count()}")
    df_verify.printSchema()

    log.info("Done!")


if __name__ == "__main__":
    main()


df = spark.read.parquet(
    "abfss://bronzelayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"
)

# df.printSchema()
# display(df)
display(df.limit(20))
# display(
#     df.select("Symbol")
#       .distinct()
#       .orderBy("Symbol")
# )

In [0]:

df = spark.read.parquet(
    "abfss://bronzelayer@cryptodl.dfs.core.windows.net/Batch_Data_Source_1/"
)

# df.printSchema()
# display(df)
display(df.limit(20))
# display(
#     df.select("Symbol")
#       .distinct()
#       .orderBy("Symbol")
# )